In [ ]:
%load_ext autoreload
%autoreload 2

from py_files.common_functions import *

########################################################################
## 발송계정 / 앱비밀번호 / 수신자는 notebooks/.env 에서 자동으로 읽어옵니다.
## 변경이 필요하면 notebooks/.env 를 수정한 뒤 이 셀을 다시 실행하세요.
## (비밀번호를 이 노트북에 직접 적지 마세요)
########################################################################
print(f"발송 계정   : {SEND_ADDR}")
print(f"수신자      : {', '.join(RECV_ADDRS)}")
print(f"다운로드 폴더: {DOWNLOAD_FOLDER}")

today = datetime.date.today().strftime('%y%m%d')
today = '250901' #yymmdd 형식으로 기재

In [3]:
browser = get_browser()
update_check(browser, today)

,번호,구분,분야,제목,일련번호,등록일
0,4151,법령해석,,새도약기금 대부업 등록대상 제외 여부(수기 건 일련번호 259001),250206,251020
1,4150,법령해석,,새도약기금 관련 양수인 평가 생략 가능 여부(수기 건 일련번호 259002),250207,251020
2,4149,비조치의견서,공통,연체이력 정보 공유 제한을 통한 서민·소상공인 신용회복지원 방안 관련 비조치의견서 ...,250046,250916
3,4148,비조치의견서,전자금융,미등록 PG 계약체결 금지 관련 비조치의견서 직권발급,250044,250910
4,4147,비조치의견서,전자금융,전자금융거래법상 선불업자 행위규칙 관련 비조치의견서 발급,250045,250910
5,4146,비조치의견서,보험,"보험업법, 금산법 상 재무건전성 유지 요건 관련 비조치 요청",250042,250903
6,4145,비조치의견서,자본시장,다자간매매체결회사의 거래량 한도규제의 비조치 가능 여부,250043,250903


새로 올라온 법령해석 정보가 있습니다. 다음 셀을 실행해주세요.


In [ ]:
move_to_first_page(browser)
last_page_num = get_last_page_num(browser)

table_list = []
sended_mails = 0
## 모든페이지를 돌면서 업데이트된 정보가 있는지 파악함
for page_num in range(1, last_page_num+1):
    if page_num!=1: ## 첫페이지는 무조건 테이블을 수집
        num_tag = get_num_tag(browser) ## 화면 하단 페이지 태그 부분을 가져와서 해당 페이지로 이동
        for a_tag in num_tag.find_elements(By.TAG_NAME, 'a'):        
            find_page_num = a_tag.text.strip()
            if find_page_num:
                if page_num==int(find_page_num):
                    a_tag.click()
                    break
    while True:
        try:
            table_df, tbody_tag = get_table_data(browser)
            break
        except:
            time.sleep(0.05)            
    table_df['등록일'] = table_df['등록일'].map(lambda x: pd.to_datetime(x).strftime("%y%m%d"))

    ## 전체 테이블과 가져올 테이블의 수가 같으면 다음페이지도 조사
    find_continue = (len(table_df)==len(table_df[table_df.등록일>=today])) 
    table_df = table_df[table_df.등록일>=today]
    if len(table_df) > 0: ## 찾아야할 테이블이 있다면
        ## 각 항목들을 클릭해서 파일을 다운로드 하고, 본문을 수집한 후 메일까지 발송
        get_page_laws_info(browser, table_df)
        sended_mails = sended_mails + len(table_df)
        table_list.append(table_df)
    if find_continue:
        time.sleep(0.3)
        continue            
    else:        
        break
print(f'총 {sended_mails} 개의 메일을 발송하였습니다. 발송 내용은 아래를 참고해주세요.')
display(pd.concat(table_list))